In [ ]:
# 4. Validação final
from src.config import loader as cfg

PERIODO = cfg.get_periodo_estudo()
START_EXPECTED = pd.to_datetime(PERIODO["start"])
END_EXPECTED = pd.to_datetime(PERIODO["end"])

min_date = df_clean['published_at'].min()
max_date = df_clean['published_at'].max()

print("\n" + "="*60)
print("✅ VALIDAÇÃO FINAL DO DATASET")
print("="*60)
print(f"\n📅 Período configurado: {START_EXPECTED.date()} → {END_EXPECTED.date()}")
print(f"📅 Período obtido:      {min_date.date()} → {max_date.date()}")
print(f"\n📊 Total de registros:  {len(df_clean):,}")
print(f"📊 Fontes únicas:       {df_clean['source'].nunique():,}")
print(f"📊 Tipos de fonte:      {df_clean['source_type'].nunique()}")

# Verificações
dias_cobertura = (max_date - min_date).days
dias_esperados = (END_EXPECTED - START_EXPECTED).days

print(f"\n🔍 Cobertura: {dias_cobertura}/{dias_esperados} dias ({(dias_cobertura/dias_esperados)*100:.1f}%)")

if len(df_clean) < 5000:
    print("⚠️ AVISO: Volume < 5.000 notícias pode limitar modelos")
elif len(df_clean) < 30000:
    print("⚠️ AVISO: Volume < 30.000 notícias (ideal para TCC)")
else:
    print("✅ Volume adequado para modelagem")

if dias_cobertura < dias_esperados * 0.7:
    print("⚠️ AVISO: Cobertura < 70% do período esperado")
else:
    print("✅ Cobertura temporal adequada")

print("\n⏭️ Próximo passo: Executar Notebook 14 (Preprocessamento PT-BR)")

In [ ]:
# 3. Visualizações e análises
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

# 3.1 Distribuição temporal
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

df_clean['published_at_date'] = pd.to_datetime(df_clean['published_at'])
df_clean['year_month'] = df_clean['published_at_date'].dt.to_period('M')

timeline = df_clean.groupby('year_month').size()
timeline.plot(ax=axes[0], title='Distribuição Temporal de Notícias', color='steelblue')
axes[0].set_xlabel('Mês')
axes[0].set_ylabel('Quantidade de Notícias')
axes[0].grid(True, alpha=0.3)

# 3.2 Distribuição por fonte
source_counts = df_clean['source_type'].value_counts()
source_counts.plot(kind='bar', ax=axes[1], title='Notícias por Tipo de Fonte', color='coral')
axes[1].set_xlabel('Tipo de Fonte')
axes[1].set_ylabel('Quantidade')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"📊 Estatísticas descritivas:")
print(df_clean[['source_type', 'source']].describe())

## 📊 Análise do Dataset Limpo

In [ ]:
# 2. Executar pipeline ETL completo
from src.pipeline.etl_pipeline import run_etl_pipeline

# Executar com dedup por embedding DESATIVADO (muito lento)
# Para ativar, use: use_embedding_dedup=True
df_clean = run_etl_pipeline(input_file=None, use_embedding_dedup=False)

print(f"\n✅ Dataset limpo disponível: {len(df_clean):,} registros")
print(f"Colunas: {list(df_clean.columns)}")

In [ ]:
# 1. Setup
import os
import sys
from pathlib import Path

# Adicionar src ao path
project_root = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
sys.path.insert(0, str(project_root))

print(f"📂 Project root: {project_root}")

# 13. ETL e Deduplicação de Notícias Multisource

**Objetivo**: Processar dados brutos do NB 12 aplicando:

1. **Deduplicação** (URL, título+data, similaridade de embeddings)
2. **Normalização** (timezone, datas, campos)
3. **Validação** (campos essenciais, mínimo de caracteres)
4. **Relatório** (estatísticas completas do ETL)

**Output**: `data_processed/news_multisource.parquet` (dataset limpo)